# Normalizing Flows — Variational Inference with Normalizing Flows (2015)

_Papers · Meta-learning, Bayesian & Robustness_

**Bend a simple Gaussian into a rich, complex density with a chain of invertible maps, tracking its exact log-density along the way.**

---

This notebook is a practice scaffold for the **Normalizing Flows — Variational Inference with Normalizing Flows (2015)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

We build up a planar normalizing flow in four steps: (1) check the single-flow change-of-variables formula by hand against autograd, (2) wrap the planar map as a module, (3) compose several flows and track the running log-density, then (4) read off how the flow reshapes a Gaussian.

### Step 1 — Verify one planar flow by hand vs autograd

A planar flow is `f(z) = z + u * tanh(w^T z + b)`. Its log-determinant has a closed form, `ln|1 + u^T psi(z)|`, where `psi(z) = (1 - tanh^2(a)) * w` (Eqns. 11-12). We compute every piece on a single 2-D point by hand, then ask `torch.autograd.functional.jacobian` for the full Jacobian and take `ln|det J|` directly. The two log-determinants should agree to many decimals — confirming the cheap closed form is exact.

In [ ]:
import torch
import torch.nn as nn
import math
import numpy as np

torch.manual_seed(0)
np.random.seed(0)

# Worked example: ONE planar flow on a 2-D point, by hand, vs autograd.
# f(z) = z + u * tanh(w^T z + b);  psi(z) = (1 - tanh^2(a)) * w;  logdet = ln|1 + u^T psi|.
z = torch.tensor([1.0, 0.5])
w = torch.tensor([1.0, -1.0])
u = torch.tensor([2.0, 0.0])
b = 0.5

a  = (w @ z) + b          # pre-activation w^T z + b = 1.0
h  = torch.tanh(a)
hp = 1 - h * h            # h'(a) = 1 - tanh^2(a)
fz = z + u * h
psi = hp * w              # psi(z) = h'(a) * w  (Eqn 11)
uTpsi = (u @ psi).item()
logdet = math.log(abs(1 + uTpsi))   # ln|1 + u^T psi|  (Eqn 12)

print("worked example: a=%.1f  tanh=%.6f  h'=%.6f" % (a.item(), h.item(), hp.item()))
print("  f(z)=%s  psi=%s" % ([round(v, 6) for v in fz.tolist()], [round(v, 6) for v in psi.tolist()]))
print("  u^T psi=%.6f   logdet=ln|1+u^T psi|=%.6f" % (uTpsi, logdet))

# ORACLE: log-determinant from the full Jacobian via autograd.
def f_one(zz):
    return zz + u * torch.tanh((w @ zz) + b)

J = torch.autograd.functional.jacobian(f_one, z.clone().requires_grad_(True))
print("  autograd ln|det J| = %.6f   (matches)" % math.log(abs(torch.det(J).item())))
# worked example: a=1.0  tanh=0.761594  h'=0.419974
#   f(z)=[2.523188, 0.5]  psi=[0.419974, -0.419974]
#   u^T psi=0.839949   logdet=ln|1+u^T psi|=0.609738
#   autograd ln|det J| = 0.609738   (matches)

### Step 2 — Wrap the planar flow as a module

Now we package the same map as an `nn.Module` with learnable parameters `w`, `u`, `b`. Its `forward` returns **both** the transformed point `f(z)` and the per-sample log-determinant `ln|1 + u^T psi|`, so the change-of-variables bookkeeping travels with the transform. Working on a batch `z` of shape `(N, dim)` keeps the same formula, just vectorised over samples.

In [ ]:
# The planar flow as an nn.Module (Eqns 10-12). Forward returns f(z) AND log|det|.
class PlanarFlow(nn.Module):
    def __init__(self, dim=2):
        super().__init__()
        self.w = nn.Parameter(torch.randn(dim) * 1.5)
        self.u = nn.Parameter(torch.randn(dim) * 1.5)
        self.b = nn.Parameter(torch.randn(1))

    def forward(self, z):                    # z: (N, dim)
        a   = z @ self.w + self.b            # (N,)  pre-activation
        h   = torch.tanh(a)
        fz  = z + h.unsqueeze(-1) * self.u   # Eqn 10
        psi = (1 - h * h).unsqueeze(-1) * self.w           # Eqn 11
        logdet = torch.log(torch.abs(1 + psi @ self.u))    # Eqn 12: ln|1 + u^T psi|
        return fz, logdet

### Step 3 — Compose K flows and track the log-density

We stack `K = 3` planar flows and push a batch of unit-Gaussian samples through them. The change-of-variables rule (Eqns. 6-7) says the log-density updates as `ln q_K = ln q_0 - sum_k ln|det f_k|`, so we **subtract** each flow's log-determinant as we go. After the chain we verify on one sample: the running `ln q_K` should match `ln q_0 - ln|det J_full|`, where `J_full` is the autograd Jacobian of the *entire* composed flow.

In [ ]:
# Compose K=3 flows; track ln q_K = ln q_0 - sum_k ln|det| (change of variables, Eqns 6-7).
torch.manual_seed(1)
K = 3
flows = nn.ModuleList([PlanarFlow(2) for _ in range(K)])

N = 50
z0 = torch.randn(N, 2)
log_q = -0.5 * (z0**2).sum(-1) - math.log(2 * math.pi)   # ln q_0 for 2-D unit Gaussian

z = z0
for flow in flows:
    z, ld = flow(z)
    log_q = log_q - ld           # SUBTRACT each flow's log-determinant

# Verify change of variables on one sample against the full-flow autograd Jacobian.
def full_flow(zz):
    cur = zz
    for flow in flows:
        cur = cur + torch.tanh(cur @ flow.w + flow.b) * flow.u
    return cur

Jf = torch.autograd.functional.jacobian(full_flow, z0[0].clone().requires_grad_(True))
lq0_0 = (-0.5 * (z0[0]**2).sum() - math.log(2 * math.pi)).item()
print("change-of-variables check (sample 0):")
print("  ln q_K via flow logdet  = %.6f" % log_q[0].item())
print("  ln q_K via autograd det = %.6f" % (lq0_0 - math.log(abs(torch.det(Jf).item()))))

### Step 4 — See the density get reshaped

Finally we measure the effect of the flow on the cloud of points. The base samples `z0` start as a round unit Gaussian (per-dim variance near 1). After the three planar maps, the per-dim variances of `z` are stretched and unequal — the round blob has been bent into a more complex, anisotropic density. That reshaping, with the exact log-density tracked alongside, is the whole point of normalizing flows.

In [ ]:
# The effect: a round Gaussian becomes a stretched / curved density.
print("base  z0   per-dim variance:", [round(v, 3) for v in z0.var(0).tolist()])
print("flowed z_K per-dim variance:", [round(v, 3) for v in z.var(0).tolist()])
# change-of-variables check (sample 0):
#   ln q_K via flow logdet  = -3.826720
#   ln q_K via autograd det = -3.826719   <- matches to 5 decimals
# base  z0   per-dim variance: [1.329, 0.709]
# flowed z_K per-dim variance: [1.491, 8.651]   <- density reshaped by the flow
# (Our small CPU run with fixed random flow params, not the paper's reported numbers.)

## Visualize the data & results

_When we push a 2-D unit Gaussian through a few planar flows, does the round blob actually deform into a more complex density — and does our tracked log-density agree with autograd?_

### Step 1 — Rebuild the flow and push points through it

This self-contained block re-defines the planar flow and composes `K = 3` of them, exactly as before, on a small batch of `N = 20` Gaussian points so the scatter is easy to read. As each flow runs we subtract its log-determinant to keep `ln q_K` current (Eqns. 6-7). The result is two clouds — the base `z0` and the flowed `z` — plus their tracked log-densities, ready to compare.

In [ ]:
import torch
import torch.nn as nn
import math
import numpy as np

torch.manual_seed(0)
np.random.seed(0)

class PlanarFlow(nn.Module):
    def __init__(self, dim=2):
        super().__init__()
        self.w = nn.Parameter(torch.randn(dim) * 1.5)
        self.u = nn.Parameter(torch.randn(dim) * 1.5)
        self.b = nn.Parameter(torch.randn(1))

    def forward(self, z):
        a = z @ self.w + self.b
        h = torch.tanh(a)
        fz = z + h.unsqueeze(-1) * self.u                  # Eqn 10
        psi = (1 - h * h).unsqueeze(-1) * self.w           # Eqn 11
        logdet = torch.log(torch.abs(1 + psi @ self.u))    # Eqn 12
        return fz, logdet

torch.manual_seed(1)
K = 3
flows = nn.ModuleList([PlanarFlow(2) for _ in range(K)])

N = 20
z0 = torch.randn(N, 2)
log_q = -0.5 * (z0**2).sum(-1) - math.log(2 * math.pi)   # ln q_0

z = z0
for flow in flows:
    z, ld = flow(z)
    log_q = log_q - ld           # Eqns 6-7

### Step 2 — Verify the density and report the reshaping

We close the loop the same way as the reference run: take the autograd Jacobian of the full composed flow on one sample and confirm the tracked `ln q_K` matches `ln q_0 - ln|det J|`. Then we print the base vs flowed per-dim variances and the actual point coordinates — the numbers a plot would draw — so you can see the round Gaussian deform into a stretched density with its exact log-density verified.

In [ ]:
# Verify change of variables on one sample vs autograd full-Jacobian.
def full(zz):
    c = zz
    for fl in flows:
        c = c + torch.tanh(c @ fl.w + fl.b) * fl.u
    return c

J = torch.autograd.functional.jacobian(full, z0[0].clone().requires_grad_(True))
lq0 = (-0.5 * (z0[0]**2).sum() - math.log(2 * math.pi)).item()
print("ln q_K flow   :", round(log_q[0].item(), 5))
print("ln q_K autograd:", round(lq0 - math.log(abs(torch.det(J).item())), 5))
print("base  var:", [round(v, 3) for v in z0.var(0).tolist()])
print("flowed var:", [round(v, 3) for v in z.var(0).tolist()])
print("base  pts:", [[round(a, 3) for a in p] for p in z0.tolist()])
print("flowed pts:", [[round(a, 3) for a in p] for p in z.tolist()])
# ln q_K flow   : -3.82672
# ln q_K autograd: -3.82672    <- change of variables verified
# base  var: [1.017, 0.822]    flowed var: [1.267, 8.096]
# Our small run with fixed random flow params, not the paper's number.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The log-determinant ablation. You have a working planar flow that tracks
            $\ln q_K = \ln q_0 - \sum_k \ln|\det \partial f_k|$ and matches autograd. You now drop the
            log-determinant term &mdash; you set $\ln q_K = \ln q_0$, as if every map had determinant 1. What
            breaks, and why is this not just a small error?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Remove the running subtraction: stop accumulating $\ln|1 + u^{\top}\psi(z)|$ per flow. — _Without the term you are claiming each map preserves volume exactly ($\det = 1$), which a planar flow does not &mdash; it stretches and squeezes space._
- Re-run the change-of-variables check against the autograd Jacobian determinant of the full flow. — _The tracked $\ln q_K$ now equals $\ln q_0$, but the true value is $\ln q_0 - \ln|\det J_{\text{full}}|$. The mismatch equals the dropped log-determinant._
- Note the consequence for VI: the density is no longer normalized, so the variational bound it feeds is invalid. — _A flow's whole value is an exact density; drop the Jacobian term and you no longer have a valid density at all._

**Answer:** Dropping the log-determinant assumes every map preserves volume ($\det = 1$), which a planar
                 flow does not. The tracked $\ln q_K$ collapses to $\ln q_0$ and stops matching the autograd
                 Jacobian determinant &mdash; the error is exactly the missing $\sum_k \ln|\det \partial f_k|$.
                 This is not a small mistake: the log-determinant is what makes the transformed density a
                 valid, normalized density. Without it the "density" does not integrate to 1, and any
                 variational bound built on it is meaningless. The Jacobian term is the flow.

</details>

**Problem 2.** Why does the planar flow have a log-determinant that costs only $O(d)$ (linear in the dimension
            $d$), when a general invertible map's Jacobian determinant costs $O(d^3)$ to compute?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Write the planar Jacobian: $\partial f / \partial z = I + u\,\psi(z)^{\top}$, the identity plus a rank-one outer product. — _The ridge term $u\,h(w^{\top}z+b)$ depends on $z$ only through the scalar $w^{\top}z$, so its derivative is the single outer product $u\,\psi(z)^{\top}$._
- Apply the matrix determinant lemma: $\det(I + u\,v^{\top}) = 1 + v^{\top}u$. — _For a rank-one update the determinant collapses to a single scalar &mdash; no $d \times d$ factorization needed._
- Count the cost: one dot product $u^{\top}\psi(z)$ is $O(d)$; one $\ln$ is $O(1)$. — _A general determinant needs an $O(d^3)$ LU factorization; the rank-one structure avoids it entirely._

**Answer:** Because the planar Jacobian is "identity plus rank-one," $I + u\,\psi(z)^{\top}$. The matrix
                 determinant lemma turns the determinant of any rank-one update into a single scalar,
                 $1 + u^{\top}\psi(z)$ &mdash; one dot product, $O(d)$, plus a logarithm. A general $d \times d$
                 Jacobian determinant needs an $O(d^3)$ factorization. The whole design of the planar (and radial)
                 flows is to pick maps whose Jacobian has special structure so the determinant stays cheap; that is
                 what lets you stack many of them.

</details>

**Problem 3.** In the worked example you had $z = (1.0, 0.5)$, $w = (1.0, -1.0)$, $u = (2.0, 0.0)$, $b = 0.5$, giving
            $a = 1.0$, $\psi(z) = (0.419974, -0.419974)$, $u^{\top}\psi = 0.839949$, and log-determinant
            $0.609738$. Now change $u$ to $u = (0.0, 2.0)$ (push along the second axis instead). Recompute
            $u^{\top}\psi(z)$ and the log-determinant. What does the result tell you?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Note $a$, $h'(a)$, and $\psi(z)$ are unchanged: they depend only on $w, b, z$, not on $u$. So $\psi(z) = (0.419974, -0.419974)$ still. — _$u$ does not enter the pre-activation $w^{\top}z + b$ or the helper $\psi$ &mdash; only the forward push and the dot product._
- Compute the new dot product: $u^{\top}\psi = (0.0)(0.419974) + (2.0)(-0.419974) = -0.839949$. — _Pushing along the second axis, where $\psi$ is negative, flips the sign of the dot product._
- Compute the log-determinant: $\ln|1 + (-0.839949)| = \ln|0.160051| = \ln(0.160051) = -1.832$. — _Now $1 + u^{\top}\psi = 0.160 compresses volume here and the log-determinant is negative._

**Answer:** With $u = (0.0, 2.0)$: $\psi(z)$ is unchanged at $(0.419974, -0.419974)$, but
                 $u^{\top}\psi = -0.839949$, so $1 + u^{\top}\psi = 0.160051$ and the log-determinant is
                 $\ln(0.160051) \approx -1.832$. The sign flipped: the original $u$ spread volume out
                 (determinant $1.84 > 1$, positive log-determinant), while this $u$ compresses it
                 (determinant $0.16  0$, so this map is still
                 invertible here.)

</details>